In [2]:
!pip install -q faiss-gpu-cu12 sentence-transformers transformers accelerate bitsandbytes fastapi uvicorn pyngrok nest-asyncio sentencepiece sacremoses googletrans==4.0.0-rc1 httpcore==0.9.1

In [14]:
import json
import re
import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, MarianMTModel, MarianTokenizer
import uvicorn
import nest_asyncio
from pyngrok import ngrok
from fastapi import FastAPI
from pydantic import BaseModel
from kaggle_secrets import UserSecretsClient

# Setup Ngrok Token from Kaggle Secrets
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_TOKEN")
ngrok.set_auth_token(ngrok_token)

# ── Crime categories — defined here so all cells can access ──
crime_categories = [
    "Homicide", "Attempted Murder", "Aggravated Assault", "Simple Assault",
    "Kidnapping", "Sexual Assault", "Domestic Violence", "Burglary",
    "Larceny/Theft", "Motor Vehicle Theft", "Arson", "Vandalism/Property Damage",
    "Trespassing", "Fraud/Deception", "Cybercrime/Hacking", "Identity Theft",
    "Extortion/Blackmail", "Embezzlement", "Drug Trafficking", "Drug Possession",
    "Weapons Offenses", "Disorderly Conduct", "Traffic/DUI", "Hit and Run",
    "Stalking", "Harassment", "Robbery"
]

print(f"✅ {len(crime_categories)} crime categories loaded")

✅ 27 crime categories loaded


In [15]:
# # ── Translation Model (Hinglish → English) ───────────────
# from googletrans import Translator
# import asyncio

# # Initialize the translator
# translator = Translator()

# def translate_to_english(text: str) -> str:
#     """
#     Converts Hinglish (Roman script) → English using Google Translate.
#     Automatically detects language so works for:
#     - Roman Hinglish  (mera account hack ho gaya)
#     - Devanagari Hindi (मेरा अकाउंट हैक हो गया)
#     - Mixed Hindi+English sentences
#     - Pure English (returned as-is)
#     """
#     try:
#         result = translator.translate(text, src='hi', dest='en')
#         return result.text
#     except Exception as e:
#         # If translation fails for any reason, return original text
#         print(f"  [Translation warning: {e}]")
#         return text

# print("✅ Translation model ready")

# # ── Quick test ────────────────────────────────────────────
# test_sentences = [
#     "Sir mera account hack ho gaya aur Rs 45,000 nikal gaye",
#     "Kisi ne mujhe UPI ke through paise chura liye",
#     "Mere ghar mein ghuskar laptop chura liya gaya",
#     "Neighbour ne mujhpe haath uthaya aur dhamki di",
#     "mere dost nee dusre dost koo chaku se marr diya",
#     "char logon ne milke ek ladki ka balaatkaar kiya"
# ]

# print()
# for t in test_sentences:
#     result = translate_to_english(t)
#     print(f"IN : {t}")
#     print(f"OUT: {result}")
#     print()


#--------------------------------------------------------------------------------------------------------------------------------------------- PHASE -2 CODE

# # ── CELL 3 : Input Processing Layer ──────────────────────
# def detect_language(text: str) -> str:
#     """
#     Detects whether input is:
#     - 'hindi'     → Devanagari script (मेरा)
#     - 'english'   → pure English with NO Hindi words
#     - 'hinglish'  → Roman script Hindi/mixed (mera, gaya, ho, etc.)
#     """
#     # Check for Devanagari script first
#     devanagari = sum(1 for c in text if '\u0900' <= c <= '\u097F')
#     if devanagari > 0:
#         return 'hindi'

#     # Common Hinglish words — if any of these appear, it is Hinglish
#     hinglish_keywords = [
#         'mera', 'meri', 'mere', 'mujhe', 'mujhse', 'maine',
#         'gaya', 'gayi', 'gaye', 'ho', 'hai', 'hain', 'tha', 'thi',
#         'aur', 'kisi', 'ne', 'se', 'ke', 'ki', 'ka', 'ko',
#         'nahi', 'nahin', 'nikaliye', 'nikal', 'liye', 'liya',
#         'kiya', 'kar', 'raha', 'rahi', 'wala', 'wali',
#         'ghus', 'chura', 'churaya', 'maara', 'maar',
#         'dhamki', 'haath', 'uthaya', 'bheja', 'aaya',
#         'paisa', 'paise', 'rupaye', 'account', 'hack',
#         'pe', 'par', 'bhi', 'sirf', 'bahut', 'kuch',
#         'sir', 'sahib', 'ji', 'aadmi', 'aurat',
#     ]

#     text_lower = text.lower()
#     words      = text_lower.split()

#     for word in words:
#         # Strip punctuation from word before checking
#         clean_word = word.strip('.,!?;:')
#         if clean_word in hinglish_keywords:
#             return 'hinglish'

#     return 'english'


# def translate_to_english(text: str) -> str:
#     """
#     Smart translation with proper language detection.
#     - Pure English  → returned as-is
#     - Devanagari    → translated
#     - Hinglish      → translated
#     """
#     text = clean_text(text)
#     lang = detect_language(text)

#     print(f"  [detected: {lang}]")

#     if lang == 'english':
#         return text  # already English, skip translation

#     try:
#         result = translator.translate(text, src='hi', dest='en')
#         return result.text
#     except Exception as e:
#         print(f"  [Translation warning: {e}]")
#         return text


# print("✅ Input processing layer updated")

# # ── Test detection ────────────────────────────────────────
# samples = [
#     "Sir mera bank account hack ho gaya aur kisi ne Rs 45,000 nikal liye",
#     "Someone broke into my house and stole my laptop",
#     "मेरे बैंक अकाउंट से पैसे निकाल लिए गए",
#     "Neighbour ne mujhpe haath uthaya aur dhamki di",
# ]

# for s in samples:
#     lang = detect_language(s)
#     translated = translate_to_english(s)
#     print(f"IN   : {s}")
#     print(f"LANG : {lang}")
#     print(f"OUT  : {translated}")
#     print()

#---------------------------------------------------------------------------------- PHASE -2 CODE 01-------------------


# ── CELL 3 : Input Processing Layer ──────────────────────
import re

try:
    from googletrans import Translator
    translator = Translator()
    print("✅ Google Translate loaded")
except Exception as e:
    print(f"⚠️ Translator failed: {e}")
    translator = None

# ── Hinglish keywords ─────────────────────────────────────
HINGLISH_KEYWORDS = [
    "mera", "meri", "mere", "kiya", "gaya", "gayi", "hai",
    "tha", "thi", "aur", "nahi", "bahut", "accha", "kya",
    "ke", "ka", "ki", "ne", "ko", "se", "pe", "par",
    "wala", "wali", "hoga", "hogi", "karo", "karna",
    "hua", "hui", "raha", "rahi", "liya", "liye",
    "ghuskar", "chura", "nikala", "lagaya", "bheja"
]

# ── clean_text ────────────────────────────────────────────
def clean_text(text: str) -> str:
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'([!?,.])\1+', r'\1', text)
    return text

# ── detect_language ───────────────────────────────────────
def detect_language(text: str) -> str:
    for ch in text:
        if '\u0900' <= ch <= '\u097F':
            return "hindi"
    words = text.lower().split()
    matches = sum(1 for w in words if w in HINGLISH_KEYWORDS)
    if matches >= 1:
        return "hinglish"
    return "english"

# ── translate_to_english ──────────────────────────────────
def translate_to_english(text: str) -> str:
    text = clean_text(text)
    lang = detect_language(text)
    print(f"  [detected: {lang}]")
    if lang == "english":
        return text
    if translator is None:
        print("  ⚠️ Translator not available, returning original")
        return text
    try:
        result = translator.translate(text, src="hi", dest="en")
        return result.text
    except Exception as e:
        print(f"  ⚠️ Translation failed: {e}")
        return text

# ── quick test ────────────────────────────────────────────
samples = [
    "Sir mera bank account hack ho gaya",
    "Someone stole my laptop",
    "मेरा पैसा चोरी हो गया",
]
for s in samples:
    lang = detect_language(s)
    translated = translate_to_english(s)
    print(f"IN  : {s}")
    print(f"LANG: {lang}")
    print(f"OUT : {translated}")
    print()

print("✅ Cell 3 ready")

✅ Google Translate loaded
  [detected: hinglish]
IN  : Sir mera bank account hack ho gaya
LANG: hinglish
OUT : sir my bank account got hacked

  [detected: english]
IN  : Someone stole my laptop
LANG: english
OUT : Someone stole my laptop

  [detected: hindi]
IN  : मेरा पैसा चोरी हो गया
LANG: hindi
OUT : my money was stolen

✅ Cell 3 ready


In [19]:
# ── CELL 4 : Knowledge Base (200 examples) ───────────────

severity_levels = ["Low", "Medium", "High", "Critical"]

knowledge_base = [

    # ── Cybercrime / Hacking (8 examples) ────────────────
    {"complaint": "Someone hacked into my email account and changed the password",                      "categories": ["Cybercrime/Hacking"],                         "severity": "High"},
    {"complaint": "My social media account was taken over by an unknown person",                        "categories": ["Cybercrime/Hacking"],                         "severity": "High"},
    {"complaint": "Unauthorized access to my company server was detected",                              "categories": ["Cybercrime/Hacking"],                         "severity": "Critical"},
    {"complaint": "Someone broke into my Gmail and is sending emails on my behalf",                     "categories": ["Cybercrime/Hacking"],                         "severity": "High"},
    {"complaint": "My Instagram was hacked and the hacker is demanding money",                          "categories": ["Cybercrime/Hacking", "Extortion/Blackmail"],  "severity": "High"},
    {"complaint": "A virus was installed on my laptop remotely and my files are encrypted",             "categories": ["Cybercrime/Hacking"],                         "severity": "Critical"},
    {"complaint": "My WhatsApp was cloned and someone is messaging my contacts pretending to be me",    "categories": ["Cybercrime/Hacking"],                         "severity": "High"},
    {"complaint": "Unknown person logged into my net banking from a different device",                  "categories": ["Cybercrime/Hacking", "Fraud/Deception"],      "severity": "Critical"},

    # ── Fraud / Deception (10 examples) ──────────────────
    {"complaint": "I received a fake UPI payment request and lost money",                               "categories": ["Fraud/Deception"],                            "severity": "High"},
    {"complaint": "Someone posed as a bank officer and took my OTP",                                    "categories": ["Fraud/Deception"],                            "severity": "High"},
    {"complaint": "Fake job offer website collected registration fees and disappeared",                 "categories": ["Fraud/Deception"],                            "severity": "Medium"},
    {"complaint": "Online shopping site took payment but never delivered the product",                  "categories": ["Fraud/Deception"],                            "severity": "Medium"},
    {"complaint": "Received a fake lottery winning message and paid processing fee",                    "categories": ["Fraud/Deception"],                            "severity": "High"},
    {"complaint": "Investment scheme promised high returns and vanished with my money",                 "categories": ["Fraud/Deception"],                            "severity": "Critical"},
    {"complaint": "A person pretending to be a government officer asked for bribe to clear my file",    "categories": ["Fraud/Deception"],                            "severity": "High"},
    {"complaint": "Fake rental website took advance for a flat that does not exist",                    "categories": ["Fraud/Deception"],                            "severity": "High"},
    {"complaint": "Received a call saying my Aadhaar will be blocked and paid fine",                    "categories": ["Fraud/Deception"],                            "severity": "High"},
    {"complaint": "Travel agency took full payment for a tour package and shut down",                   "categories": ["Fraud/Deception"],                            "severity": "Medium"},

    # ── Identity Theft (6 examples) ───────────────────────
    {"complaint": "Someone used my Aadhaar card to open a bank account without my knowledge",           "categories": ["Identity Theft"],                             "severity": "Critical"},
    {"complaint": "My PAN card details were misused to file a fake income tax return",                  "categories": ["Identity Theft"],                             "severity": "Critical"},
    {"complaint": "Someone created a fake profile using my photos and name on Facebook",                "categories": ["Identity Theft"],                             "severity": "High"},
    {"complaint": "My credit card was cloned and used for purchases in another city",                   "categories": ["Identity Theft", "Fraud/Deception"],          "severity": "High"},
    {"complaint": "A loan was taken in my name using my documents without my consent",                  "categories": ["Identity Theft"],                             "severity": "Critical"},
    {"complaint": "Someone submitted a SIM swap request using my documents and took my number",         "categories": ["Identity Theft", "Cybercrime/Hacking"],       "severity": "Critical"},

    # ── Harassment (8 examples) ───────────────────────────
    {"complaint": "Receiving continuous abusive messages from an unknown number",                       "categories": ["Harassment"],                                 "severity": "Medium"},
    {"complaint": "My colleague is sending me inappropriate messages at work",                          "categories": ["Harassment"],                                 "severity": "Medium"},
    {"complaint": "Someone is spreading false rumors about me on social media",                         "categories": ["Harassment"],                                 "severity": "Medium"},
    {"complaint": "Getting threatening phone calls late at night from unknown person",                  "categories": ["Harassment"],                                 "severity": "High"},
    {"complaint": "My ex partner keeps sending unwanted messages and showing up at my workplace",       "categories": ["Harassment", "Stalking"],                     "severity": "High"},
    {"complaint": "A group of people are trolling and bullying me on every social media post",          "categories": ["Harassment"],                                 "severity": "Medium"},
    {"complaint": "My landlord is harassing me with daily calls to vacate before lease ends",           "categories": ["Harassment"],                                 "severity": "Medium"},
    {"complaint": "Boss is sending inappropriate late night messages and making uncomfortable remarks", "categories": ["Harassment"],                                 "severity": "High"},

    # ── Stalking (6 examples) ─────────────────────────────
    {"complaint": "A person follows me every day from my home to my office",                            "categories": ["Stalking"],                                   "severity": "High"},
    {"complaint": "Someone is tracking my location without my permission using an app",                 "categories": ["Stalking"],                                   "severity": "High"},
    {"complaint": "Unknown person keeps appearing wherever I go and watching me",                       "categories": ["Stalking"],                                   "severity": "High"},
    {"complaint": "My online accounts are being monitored and screenshots shared by my ex",             "categories": ["Stalking", "Cybercrime/Hacking"],              "severity": "High"},
    {"complaint": "A man waits outside my building every morning and follows my auto",                  "categories": ["Stalking"],                                   "severity": "High"},
    {"complaint": "Receiving anonymous letters with details of my daily routine",                       "categories": ["Stalking"],                                   "severity": "High"},

    # ── Extortion / Blackmail (6 examples) ───────────────
    {"complaint": "Someone has my private photos and is demanding money to not share them",             "categories": ["Extortion/Blackmail"],                        "severity": "Critical"},
    {"complaint": "Received a message threatening to harm my family unless I pay two lakhs",            "categories": ["Extortion/Blackmail"],                        "severity": "Critical"},
    {"complaint": "Former employee threatening to leak company data unless paid",                       "categories": ["Extortion/Blackmail"],                        "severity": "Critical"},
    {"complaint": "Person threatening to post morphed images of me on social media",                    "categories": ["Extortion/Blackmail"],                        "severity": "Critical"},
    {"complaint": "Gangsters are demanding weekly protection money from my shop",                       "categories": ["Extortion/Blackmail"],                        "severity": "Critical"},
    {"complaint": "Someone recorded a private video and is now asking for money to delete it",          "categories": ["Extortion/Blackmail"],                        "severity": "Critical"},

    # ── Domestic Violence (7 examples) ────────────────────
    {"complaint": "My husband beats me regularly and I have visible injuries on my body",               "categories": ["Domestic Violence"],                          "severity": "Critical"},
    {"complaint": "My father in law physically assaults me over dowry demands",                         "categories": ["Domestic Violence"],                          "severity": "Critical"},
    {"complaint": "My wife is being mentally tortured by in-laws every single day",                     "categories": ["Domestic Violence"],                          "severity": "High"},
    {"complaint": "Spouse locked me inside the house and confiscated my phone",                         "categories": ["Domestic Violence"],                          "severity": "High"},
    {"complaint": "My husband throws things at me and threatens to kill me during fights",              "categories": ["Domestic Violence"],                          "severity": "Critical"},
    {"complaint": "In-laws are forcing me to bring more dowry and threatening consequences",            "categories": ["Domestic Violence"],                          "severity": "Critical"},
    {"complaint": "My son beats his wife and she has been hospitalised twice this year",                "categories": ["Domestic Violence"],                          "severity": "Critical"},

    # ── Sexual Assault (5 examples) ───────────────────────
    {"complaint": "I was sexually assaulted by a colleague after an office party",                      "categories": ["Sexual Assault"],                             "severity": "Critical"},
    {"complaint": "A man touched me inappropriately in a crowded bus during commute",                   "categories": ["Sexual Assault"],                             "severity": "High"},
    {"complaint": "Received unsolicited obscene images and videos from an unknown person",              "categories": ["Sexual Assault"],                             "severity": "Medium"},
    {"complaint": "A teacher made inappropriate physical contact with a student in class",              "categories": ["Sexual Assault"],                             "severity": "Critical"},
    {"complaint": "I was followed into a lift and sexually harassed by a building resident",            "categories": ["Sexual Assault"],                             "severity": "Critical"},

    # ── Burglary (6 examples) ─────────────────────────────
    {"complaint": "My house was broken into while I was away and all valuables were stolen",            "categories": ["Burglary"],                                   "severity": "High"},
    {"complaint": "Someone broke the lock of my shop at night and stole the day's cash",                "categories": ["Burglary"],                                   "severity": "High"},
    {"complaint": "Thieves entered my flat through the window and took all jewelry",                    "categories": ["Burglary"],                                   "severity": "High"},
    {"complaint": "My office was burglarized and all computers and hard drives were stolen",            "categories": ["Burglary"],                                   "severity": "High"},
    {"complaint": "While I was on vacation someone broke into my house and ransacked it",               "categories": ["Burglary"],                                   "severity": "High"},
    {"complaint": "The store room on the terrace was broken into and equipment was stolen",             "categories": ["Burglary"],                                   "severity": "Medium"},

    # ── Larceny / Theft (7 examples) ─────────────────────
    {"complaint": "My mobile phone was snatched from my hand while I was walking",                      "categories": ["Larceny/Theft"],                              "severity": "Medium"},
    {"complaint": "My wallet was pickpocketed in the metro station during rush hour",                   "categories": ["Larceny/Theft"],                              "severity": "Medium"},
    {"complaint": "Someone stole my laptop bag from the cafe while I was not looking",                  "categories": ["Larceny/Theft"],                              "severity": "Medium"},
    {"complaint": "My bicycle was stolen from outside my building overnight",                           "categories": ["Larceny/Theft"],                              "severity": "Low"},
    {"complaint": "Cash and gold chain stolen from my bag on the bus",                                  "categories": ["Larceny/Theft"],                              "severity": "Medium"},
    {"complaint": "My airpods and watch were stolen from my gym locker",                                "categories": ["Larceny/Theft"],                              "severity": "Low"},
    {"complaint": "Delivery package worth ten thousand rupees was stolen from my doorstep",             "categories": ["Larceny/Theft"],                              "severity": "Medium"},

    # ── Motor Vehicle Theft (6 examples) ─────────────────
    {"complaint": "My car was stolen from the parking lot outside the mall",                            "categories": ["Motor Vehicle Theft"],                        "severity": "High"},
    {"complaint": "My motorcycle went missing from outside my house last night",                        "categories": ["Motor Vehicle Theft"],                        "severity": "High"},
    {"complaint": "Someone broke the car window and stole items and documents from inside",             "categories": ["Motor Vehicle Theft"],                        "severity": "Medium"},
    {"complaint": "My electric scooter was taken from the building basement parking",                   "categories": ["Motor Vehicle Theft"],                        "severity": "High"},
    {"complaint": "My truck was stolen from the highway dhaba where I stopped to eat",                  "categories": ["Motor Vehicle Theft"],                        "severity": "Critical"},
    {"complaint": "Auto rickshaw was stolen from the stand where I parked it at night",                 "categories": ["Motor Vehicle Theft"],                        "severity": "High"},

    # ── Robbery (6 examples) ──────────────────────────────
    {"complaint": "A group of men stopped my car and robbed me at knifepoint on the highway",          "categories": ["Robbery"],                                    "severity": "Critical"},
    {"complaint": "Someone snatched my gold chain while I was walking near the market",                "categories": ["Robbery"],                                    "severity": "High"},
    {"complaint": "Armed men entered my shop and took all the cash from the counter",                   "categories": ["Robbery"],                                    "severity": "Critical"},
    {"complaint": "Was robbed at gunpoint near the ATM while withdrawing cash at night",                "categories": ["Robbery"],                                    "severity": "Critical"},
    {"complaint": "Two men on a bike grabbed my purse and sped away",                                   "categories": ["Robbery"],                                    "severity": "High"},
    {"complaint": "A man threatened me with a rod and took my phone and wallet",                        "categories": ["Robbery"],                                    "severity": "High"},

    # ── Kidnapping (6 examples) ───────────────────────────
    {"complaint": "My child did not return from school and phone is switched off",                      "categories": ["Kidnapping"],                                 "severity": "Critical"},
    {"complaint": "My brother was abducted by unknown persons who are demanding ransom",                "categories": ["Kidnapping"],                                 "severity": "Critical"},
    {"complaint": "A woman was forcibly pushed into a car near the market by two men",                  "categories": ["Kidnapping"],                                 "severity": "Critical"},
    {"complaint": "My employee went missing after receiving suspicious calls yesterday",                "categories": ["Kidnapping"],                                 "severity": "Critical"},
    {"complaint": "My daughter was lured away by a stranger offering a job",                            "categories": ["Kidnapping"],                                 "severity": "Critical"},
    {"complaint": "An elderly person from our society has been missing for two days",                   "categories": ["Kidnapping"],                                 "severity": "Critical"},

    # ── Homicide (4 examples) ─────────────────────────────
    {"complaint": "Dead body found near the railway tracks with multiple injury marks",                 "categories": ["Homicide"],                                   "severity": "Critical"},
    {"complaint": "My brother was murdered by his business partner over a property dispute",            "categories": ["Homicide"],                                   "severity": "Critical"},
    {"complaint": "Body of a young woman was found in the river near our village",                      "categories": ["Homicide"],                                   "severity": "Critical"},
    {"complaint": "A man was stabbed multiple times outside a bar and died on the spot",                "categories": ["Homicide"],                                   "severity": "Critical"},

    # ── Attempted Murder (4 examples) ────────────────────
    {"complaint": "Someone attacked me with a knife outside my house last night",                       "categories": ["Attempted Murder"],                           "severity": "Critical"},
    {"complaint": "My neighbor shot at me during a property dispute and I narrowly escaped",            "categories": ["Attempted Murder"],                           "severity": "Critical"},
    {"complaint": "A person tried to run me over with a car deliberately three times",                  "categories": ["Attempted Murder"],                           "severity": "Critical"},
    {"complaint": "Poison was mixed in my food at a family function and I was hospitalised",            "categories": ["Attempted Murder"],                           "severity": "Critical"},

    # ── Aggravated Assault (5 examples) ──────────────────
    {"complaint": "A group of men beat me with iron rods and I am currently hospitalised",              "categories": ["Aggravated Assault"],                         "severity": "Critical"},
    {"complaint": "I was attacked with a sharp weapon during a road rage incident",                     "categories": ["Aggravated Assault"],                         "severity": "Critical"},
    {"complaint": "My face was disfigured in an acid attack by my ex boyfriend",                        "categories": ["Aggravated Assault"],                         "severity": "Critical"},
    {"complaint": "A man hit me on the head with a heavy object and I lost consciousness",              "categories": ["Aggravated Assault"],                         "severity": "Critical"},
    {"complaint": "I was beaten badly by a mob for no reason and have multiple fractures",              "categories": ["Aggravated Assault"],                         "severity": "Critical"},

    # ── Simple Assault (5 examples) ───────────────────────
    {"complaint": "My neighbor slapped me during an argument over parking space",                       "categories": ["Simple Assault"],                             "severity": "Medium"},
    {"complaint": "A shopkeeper pushed and hit me when I asked for a refund",                           "categories": ["Simple Assault"],                             "severity": "Medium"},
    {"complaint": "I was punched by a stranger at a public event without any reason",                   "categories": ["Simple Assault"],                             "severity": "Medium"},
    {"complaint": "A passenger in the bus hit me after an argument over a seat",                        "categories": ["Simple Assault"],                             "severity": "Medium"},
    {"complaint": "My colleague shoved me during a heated argument at the office",                      "categories": ["Simple Assault"],                             "severity": "Low"},

    # ── Drug Trafficking (6 examples) ────────────────────
    {"complaint": "I noticed suspicious drug dealing activity near my building daily",                  "categories": ["Drug Trafficking"],                           "severity": "High"},
    {"complaint": "A person is supplying drugs to college students near the campus gate",               "categories": ["Drug Trafficking"],                           "severity": "Critical"},
    {"complaint": "Large quantity of drugs was found hidden in a vehicle near my area",                 "categories": ["Drug Trafficking"],                           "severity": "Critical"},
    {"complaint": "My son was approached by drug dealers outside the school",                           "categories": ["Drug Trafficking"],                           "severity": "Critical"},
    {"complaint": "A house in our lane is being used as a drug distribution point",                     "categories": ["Drug Trafficking"],                           "severity": "Critical"},
    {"complaint": "Courier packages containing drugs are being regularly delivered to my neighbour",    "categories": ["Drug Trafficking"],                           "severity": "High"},

    # ── Drug Possession (4 examples) ─────────────────────
    {"complaint": "My tenant was found with a small quantity of drugs in the rented room",              "categories": ["Drug Possession"],                            "severity": "Medium"},
    {"complaint": "A student was caught with marijuana at the college hostel",                          "categories": ["Drug Possession"],                            "severity": "Medium"},
    {"complaint": "My son was found with drugs in his school bag",                                      "categories": ["Drug Possession"],                            "severity": "High"},
    {"complaint": "A person was found unconscious with drug paraphernalia in the park",                 "categories": ["Drug Possession"],                            "severity": "Medium"},

    # ── Arson (5 examples) ────────────────────────────────
    {"complaint": "Someone set fire to my car parked outside my house last night",                      "categories": ["Arson"],                                      "severity": "Critical"},
    {"complaint": "My shop was burned down by unknown people in the middle of the night",               "categories": ["Arson"],                                      "severity": "Critical"},
    {"complaint": "A fire was deliberately started in the warehouse near our society",                  "categories": ["Arson"],                                      "severity": "Critical"},
    {"complaint": "Miscreants set fire to the crops in my field causing huge loss",                     "categories": ["Arson"],                                      "severity": "High"},
    {"complaint": "Someone threw a petrol bomb at the entrance of our office building",                 "categories": ["Arson"],                                      "severity": "Critical"},

    # ── Vandalism / Property Damage (5 examples) ─────────
    {"complaint": "My car windshield was smashed by unknown persons last night",                        "categories": ["Vandalism/Property Damage"],                  "severity": "Medium"},
    {"complaint": "Someone spray painted abusive words on the walls of my house",                       "categories": ["Vandalism/Property Damage"],                  "severity": "Low"},
    {"complaint": "The gate and compound wall of my property were deliberately damaged",                "categories": ["Vandalism/Property Damage"],                  "severity": "Medium"},
    {"complaint": "All four tyres of my car were slashed and mirrors broken",                           "categories": ["Vandalism/Property Damage"],                  "severity": "Medium"},
    {"complaint": "My shop shutters were damaged and signboard destroyed by a group",                   "categories": ["Vandalism/Property Damage"],                  "severity": "Medium"},

    # ── Trespassing (5 examples) ──────────────────────────
    {"complaint": "Unknown person keeps entering my property without permission repeatedly",            "categories": ["Trespassing"],                                "severity": "Medium"},
    {"complaint": "A group of people forcibly occupied my vacant plot claiming ownership",              "categories": ["Trespassing"],                                "severity": "High"},
    {"complaint": "Someone entered my flat while I was away and rummaged through things",               "categories": ["Trespassing"],                                "severity": "High"},
    {"complaint": "My ex keeps coming to my house uninvited despite being told not to",                "categories": ["Trespassing", "Harassment"],                  "severity": "High"},
    {"complaint": "Labourers entered my farm and started working without any permission",               "categories": ["Trespassing"],                                "severity": "Medium"},

    # ── Embezzlement (5 examples) ─────────────────────────
    {"complaint": "My accountant transferred company funds to his personal account",                    "categories": ["Embezzlement"],                               "severity": "Critical"},
    {"complaint": "The society treasurer misused the maintenance funds collected from residents",       "categories": ["Embezzlement"],                               "severity": "High"},
    {"complaint": "An employee was siphoning petty cash from the office for several months",           "categories": ["Embezzlement"],                               "severity": "High"},
    {"complaint": "The NGO director used donation money for personal expenses",                         "categories": ["Embezzlement"],                               "severity": "Critical"},
    {"complaint": "Our business partner withdrew partnership funds without any agreement",              "categories": ["Embezzlement"],                               "severity": "Critical"},

    # ── Weapons Offenses (5 examples) ────────────────────
    {"complaint": "My neighbor was found carrying an unlicensed pistol in his vehicle",                "categories": ["Weapons Offenses"],                           "severity": "Critical"},
    {"complaint": "A person threatened me by showing a country made revolver",                          "categories": ["Weapons Offenses"],                           "severity": "Critical"},
    {"complaint": "Illegal firearms were found stored in the adjacent shop",                            "categories": ["Weapons Offenses"],                           "severity": "Critical"},
    {"complaint": "A man was seen carrying a sharp sword openly on a busy street",                      "categories": ["Weapons Offenses"],                           "severity": "High"},
    {"complaint": "Crude bombs were found buried in a field near the village",                          "categories": ["Weapons Offenses"],                           "severity": "Critical"},

    # ── Disorderly Conduct (5 examples) ──────────────────
    {"complaint": "A drunk man is creating nuisance outside my house every night",                      "categories": ["Disorderly Conduct"],                         "severity": "Low"},
    {"complaint": "Group of people playing loud music and disturbing the entire neighbourhood",         "categories": ["Disorderly Conduct"],                         "severity": "Low"},
    {"complaint": "A person is publicly abusing and threatening passersby on the street",               "categories": ["Disorderly Conduct"],                         "severity": "Medium"},
    {"complaint": "Men are gambling openly in the park and creating disturbance",                       "categories": ["Disorderly Conduct"],                         "severity": "Low"},
    {"complaint": "A group is blocking the road and fighting loudly since morning",                     "categories": ["Disorderly Conduct"],                         "severity": "Medium"},

    # ── Traffic / DUI (5 examples) ────────────────────────
    {"complaint": "A driver was caught driving under the influence near the school zone",               "categories": ["Traffic/DUI"],                                "severity": "High"},
    {"complaint": "Someone is racing vehicles on the residential road late at night",                   "categories": ["Traffic/DUI"],                                "severity": "Medium"},
    {"complaint": "Truck driver was driving recklessly and almost caused a major accident",             "categories": ["Traffic/DUI"],                                "severity": "High"},
    {"complaint": "A drunk bus driver was driving with passengers and swerving dangerously",            "categories": ["Traffic/DUI"],                                "severity": "Critical"},
    {"complaint": "A teenager without licence was driving at high speed in a residential area",         "categories": ["Traffic/DUI"],                                "severity": "High"},

    # ── Hit and Run (6 examples) ──────────────────────────
    {"complaint": "A car hit my father on the road and drove away without stopping",                    "categories": ["Hit and Run"],                                "severity": "Critical"},
    {"complaint": "My motorcycle was hit by a speeding vehicle that fled the scene immediately",        "categories": ["Hit and Run"],                                "severity": "High"},
    {"complaint": "An old woman was knocked down by a car and the driver escaped",                      "categories": ["Hit and Run"],                                "severity": "Critical"},
    {"complaint": "My parked car was badly damaged by another vehicle that drove off",                  "categories": ["Hit and Run"],                                "severity": "Medium"},
    {"complaint": "A cyclist was hit by a truck and the driver ran away from the spot",                 "categories": ["Hit and Run"],                                "severity": "Critical"},
    {"complaint": "A child was knocked down outside school and the driver sped away",                   "categories": ["Hit and Run"],                                "severity": "Critical"},
]

# ── Rebuild FAISS index ───────────────────────────────────
kb_embeddings = embedder.encode([kb["complaint"] for kb in knowledge_base])
index         = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(np.array(kb_embeddings))

print(f"✅ Knowledge base loaded  : {len(knowledge_base)} examples")
print(f"✅ FAISS index rebuilt    : {index.ntotal} vectors")
print(f"✅ Categories covered     : {len(set(c for kb in knowledge_base for c in kb['categories']))}")
print(f"✅ Severity levels        : {severity_levels}")

✅ Knowledge base loaded  : 156 examples
✅ FAISS index rebuilt    : 156 vectors
✅ Categories covered     : 27
✅ Severity levels        : ['Low', 'Medium', 'High', 'Critical']


In [24]:
# 3. Rebuild FAISS Index (Fixes the IndexError)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
texts = [item["complaint"] for item in knowledge_base]
embeddings = embedder.encode(texts)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# 4. Load Open Model
model_id = "HuggingFaceH4/zephyr-7b-beta"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 5. Classification Logic -------------------------------------------------------------------------------------------------


# def classify_complaint(new_complaint, top_k=2):
#     new_embed = embedder.encode([new_complaint])
#     distances, indices = index.search(np.array(new_embed), top_k)
    
#     few_shot_context = ""
#     for idx in indices[0]:
#         # Fix: Ignore out-of-bounds or -1 indices
#         if idx == -1 or idx >= len(knowledge_base):
#             continue
            
#         match = knowledge_base[idx]
#         example_json = json.dumps({"categories": match['categories'], "severity": match['severity']})
#         few_shot_context += f"Complaint: {match['complaint']}\nOutput: {example_json}\n\n"

#     system_prompt = f"""<|system|>
# You are a strict law enforcement AI. Classify the complaint into one or more categories: {json.dumps(crime_categories)}.
# Assign a severity: {json.dumps(severity_levels)}.
# Output ONLY a valid JSON object. Do not include markdown formatting, explanations, or introductory text.</s>
# <|user|>
# Examples:
# {few_shot_context}
# New Complaint: {new_complaint}
# Output:</s>
# <|assistant|>"""

#     inputs = tokenizer(system_prompt, return_tensors="pt").to("cuda")
#     outputs = model.generate(
#         **inputs, 
#         max_new_tokens=60, 
#         do_sample=True, 
#         temperature=0.1,
#         pad_token_id=tokenizer.eos_token_id
#     )
    
#     response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
#     try:
#         json_match = re.search(r'\{.*?\}', response, re.DOTALL)
#         if json_match:
#             return json.loads(json_match.group(0))
#         return {"error": "Could not parse JSON", "raw_output": response}
#     except json.JSONDecodeError:
#         return {"error": "Invalid JSON generated", "raw_output": response}

# # 6. Test
# test_text = "Someone broke into my house while I was sleeping and stole my laptop and ₹50,000 in cash."
# print(classify_complaint(test_text))

#------------------------------------------------------------------------------------------------------- IMPROVED PROMPT FOR CLASSIFICATION ACCURACY ------

def classify_complaint(english_text: str) -> dict:
    """
    Classifies a complaint using improved prompt engineering.
    Returns { categories: [...], severity: "..." }
    """

    # ── Step 1: RAG retrieval — find closest examples ─────
    query_embedding        = embedder.encode([english_text])
    distances, indices     = index.search(np.array(query_embedding), 3)
    
    # Build retrieved examples string for the prompt
    retrieved_examples = ""
    for i, idx in enumerate(indices[0]):
        ex = knowledge_base[idx]
        retrieved_examples += f"  Example {i+1}: \"{ex['complaint']}\" → {ex['categories']} | {ex['severity']}\n"

    # ── Step 2: Build the improved prompt ─────────────────
    categories_list = "\n".join([f"  - {c}" for c in crime_categories])

    prompt = f"""<|system|>
You are a professional Indian cybercrime classification officer with expertise in the Bharatiya Nyaya Sanhita (BNS) 2024 and IT Act 2000. Your job is to classify complaints into the correct crime categories and assign severity levels.

AVAILABLE CRIME CATEGORIES:
{categories_list}

SEVERITY LEVELS AND THEIR MEANING:
  - Critical : Life threat, ongoing attack, child involved, national security
  - High     : Financial loss above Rs 10000, hacking, identity theft, assault
  - Medium   : Harassment, minor fraud, cyberbullying, small theft
  - Low      : Nuisance, minor dispute, non-urgent complaints

CLASSIFICATION RULES — follow these strictly:
  1. Always pick the MOST SPECIFIC category. Example: "UPI fraud" → Fraud/Deception, NOT Cybercrime/Hacking
  2. You may assign UP TO 2 categories maximum if the complaint clearly involves two crimes
  3. If complaint mentions financial loss → always include Fraud/Deception
  4. If complaint mentions hacking or account takeover → always include Cybercrime/Hacking
  5. If complaint mentions physical violence → always include Assault or Domestic Violence
  6. If complaint mentions threats → include Harassment or Extortion/Blackmail
  7. Default to the single most relevant category — do not over-classify

SIMILAR CASES FROM DATABASE FOR REFERENCE:
{retrieved_examples}

OUTPUT FORMAT — respond ONLY with valid JSON, nothing else:
{{"categories": ["Category1"], "severity": "High"}}

OR if two categories apply:
{{"categories": ["Category1", "Category2"], "severity": "High"}}
<|end|>
<|user|>
Classify this complaint: "{english_text}"

Think step by step:
1. What is the main crime being described?
2. Is there a secondary crime involved?
3. What is the financial or physical impact?
4. What severity does this deserve?

Now output only the JSON classification:
<|end|>
<|assistant|>
{{"""

    # ── Step 3: Generate with Zephyr ──────────────────────
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.1,       # very low = more deterministic, less random
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )

    # ── Step 4: Parse the response ────────────────────────
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    # Clean up the response
    generated = "{" + generated.split("{")[-1]  if "{" not in generated[:5] else generated
    generated = generated.split("}")[0] + "}"

    try:
        # Try direct JSON parse first
        result = json.loads(generated)
    except:
        try:
            # Try extracting JSON from messy output
            import re
            json_match = re.search(r'\{[^}]+\}', generated)
            if json_match:
                result = json.loads(json_match.group())
            else:
                raise ValueError("No JSON found")
        except:
            # Fallback — use RAG result directly without LLM
            print(f"  ⚠️ JSON parse failed, using RAG fallback")
            fallback_idx  = indices[0][0]
            fallback      = knowledge_base[fallback_idx]
            result = {
                "categories": fallback["categories"],
                "severity":   fallback["severity"]
            }

    # ── Step 5: Validate the output ───────────────────────
    # Make sure categories are valid
    valid_categories = [
        c for c in result.get("categories", [])
        if c in crime_categories
    ]
    if not valid_categories:
        # If LLM returned invalid category names, use RAG fallback
        fallback_idx   = indices[0][0]
        valid_categories = knowledge_base[fallback_idx]["categories"]

    valid_severities = ["Critical", "High", "Medium", "Low"]
    severity = result.get("severity", "Medium")
    if severity not in valid_severities:
        severity = "Medium"

    return {
        "categories": valid_categories[:2],  # max 2 categories
        "severity":   severity
    }

print("✅ classify_complaint() loaded with improved prompt")

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

✅ classify_complaint() loaded with improved prompt


In [25]:
# ── CELL 6 : Automation Layer ─────────────────────────────
from datetime import datetime
import uuid


# ── BNS Section mapping (effective 1 July 2024) ───────────
# Sources: Official BNS text + IPC-to-BNS conversion tables
# IT Act sections remain unchanged (BNS did not replace IT Act)

BNS_SECTIONS = {
    "Homicide": [
        "BNS 101 - Murder",                          # was IPC 302
        "BNS 105 - Culpable Homicide",               # was IPC 304
    ],
    "Attempted Murder": [
        "BNS 109 - Attempt to Murder",               # was IPC 307
    ],
    "Aggravated Assault": [
        "BNS 117 - Grievous Hurt",                   # was IPC 325
        "BNS 118(3) - Grievous Hurt by Weapon",      # was IPC 326
    ],
    "Simple Assault": [
        "BNS 115 - Voluntarily Causing Hurt",        # was IPC 323
        "BNS 130 - Assault",                         # was IPC 351
    ],
    "Kidnapping": [
        "BNS 137 - Kidnapping",                      # was IPC 363
        "BNS 140 - Kidnapping for Ransom",           # was IPC 364A
    ],
    "Sexual Assault": [
        "BNS 64 - Rape",                             # was IPC 376
        "BNS 74 - Assault on Woman",                 # was IPC 354
        "BNS 70 - Gang Rape",                        # was IPC 376D
    ],
    "Domestic Violence": [
        "BNS 85 - Cruelty by Husband/Relatives",     # was IPC 498A
        "BNS 86 - Definition of Cruelty",
        "Protection of Women from DV Act 2005",      # unchanged
    ],
    "Burglary": [
        "BNS 305 - Theft in Dwelling House",         # was IPC 380
        "BNS 331 - House Trespass",                  # was IPC 449
    ],
    "Larceny/Theft": [
        "BNS 303 - Theft",                           # was IPC 379
        "BNS 304 - Snatching",                       # NEW in BNS
    ],
    "Motor Vehicle Theft": [
        "BNS 303 - Theft",                           # was IPC 379
        "BNS 112 - Petty Organised Crime",           # NEW in BNS - vehicle theft by gangs
        "Motor Vehicles Act 1988",                   # unchanged
    ],
    "Arson": [
        "BNS 324 - Mischief by Fire",                # was IPC 435
        "BNS 325 - Mischief by Fire on Dwelling",    # was IPC 436
    ],
    "Vandalism/Property Damage": [
        "BNS 324 - Mischief causing Damage",         # was IPC 427
    ],
    "Trespassing": [
        "BNS 329(3) - Criminal Trespass",            # was IPC 447
        "BNS 329(4) - House Trespass",               # was IPC 448
    ],
    "Fraud/Deception": [
        "BNS 318 - Cheating",                        # was IPC 420
        "BNS 318(4) - Cheating with Delivery",       # was IPC 420 (famous '420')
    ],
    "Cybercrime/Hacking": [
        "IT Act Section 66 - Hacking",               # IT Act unchanged
        "IT Act Section 43 - Damage to Computer",    # IT Act unchanged
        "BNS 111 - Organised Cybercrime",            # NEW in BNS
        "BNS 318 - Online Cheating/Fraud",           # was IPC 420
    ],
    "Identity Theft": [
        "IT Act Section 66C - Identity Theft",       # IT Act unchanged
        "BNS 319 - Cheating by Impersonation",       # was IPC 419
    ],
    "Extortion/Blackmail": [
        "BNS 308 - Extortion",                       # was IPC 383
        "BNS 309 - Punishment for Extortion",        # was IPC 384
        "BNS 288 - Extortion by Fear of Death",      # was IPC 386
    ],
    "Embezzlement": [
        "BNS 316 - Criminal Breach of Trust",        # was IPC 406
        "BNS 314 - Dishonest Misappropriation",      # was IPC 403
    ],
    "Drug Trafficking": [
        "NDPS Act Section 20 - Production/Sale",     # unchanged
        "NDPS Act Section 37 - Bail Provisions",     # unchanged
        "BNS 111 - Organised Drug Crime",            # NEW in BNS
    ],
    "Drug Possession": [
        "NDPS Act Section 27 - Possession",          # unchanged
    ],
    "Weapons Offenses": [
        "Arms Act Section 25 - Unlicensed Arms",     # unchanged
        "BNS 109 - Attempt to Murder with Weapon",   # was IPC 307
    ],
    "Disorderly Conduct": [
        "BNS 223 - Public Nuisance",                 # was IPC 268
        "BNS 290 - Punishment for Nuisance",         # was IPC 290
    ],
    "Traffic/DUI": [
        "Motor Vehicles Act Section 185 - DUI",      # unchanged
        "BNS 281 - Rash Driving",                    # was IPC 279
        "BNS 106 - Death by Negligence",             # was IPC 304A
    ],
    "Hit and Run": [
        "Motor Vehicles Act Section 161 - Hit & Run",# unchanged
        "BNS 106 - Causing Death by Negligence",     # was IPC 304A
        "BNS 281 - Rash Driving on Public Way",      # was IPC 279
    ],
    "Stalking": [
        "BNS 78 - Stalking",                         # was IPC 354D
    ],
    "Harassment": [
        "BNS 351 - Criminal Intimidation",           # was IPC 506
        "BNS 352 - Intentional Insult",              # was IPC 504
        "BNS 75 - Sexual Harassment",                # was IPC 354A
    ],
    "Robbery": [
        "BNS 309 - Robbery",                         # was IPC 392
        "BNS 292 - Robbery",                         # was IPC 390
        "BNS 296 - Robbery with Hurt",               # was IPC 394
    ],
}

# Keep IPC as fallback for reference (pre-July 2024 cases)
IPC_SECTIONS = {
    "Homicide":              ["IPC 302 - Murder", "IPC 304 - Culpable Homicide"],
    "Attempted Murder":      ["IPC 307 - Attempt to Murder"],
    "Aggravated Assault":    ["IPC 325 - Grievous Hurt", "IPC 326 - Dangerous Weapons"],
    "Simple Assault":        ["IPC 323 - Voluntarily Causing Hurt"],
    "Kidnapping":            ["IPC 363 - Kidnapping", "IPC 364A - Kidnapping for Ransom"],
    "Sexual Assault":        ["IPC 376 - Rape", "IPC 354 - Assault on Woman"],
    "Domestic Violence":     ["IPC 498A - Cruelty by Husband", "DV Act 2005"],
    "Burglary":              ["IPC 457 - Lurking House Trespass", "IPC 380 - Theft in Dwelling"],
    "Larceny/Theft":         ["IPC 379 - Theft", "IPC 381 - Theft by Clerk"],
    "Motor Vehicle Theft":   ["IPC 379 - Theft", "MV Act 1988"],
    "Arson":                 ["IPC 435 - Mischief by Fire", "IPC 436 - House Arson"],
    "Vandalism/Property Damage": ["IPC 427 - Mischief causing Damage"],
    "Trespassing":           ["IPC 447 - Criminal Trespass"],
    "Fraud/Deception":       ["IPC 420 - Cheating", "IPC 415 - Cheating Definition"],
    "Cybercrime/Hacking":    ["IT Act 66 - Hacking", "IT Act 43 - Damage to Computer"],
    "Identity Theft":        ["IT Act 66C - Identity Theft", "IPC 419 - Impersonation"],
    "Extortion/Blackmail":   ["IPC 383 - Extortion", "IPC 384 - Punishment for Extortion"],
    "Embezzlement":          ["IPC 406 - Criminal Breach of Trust"],
    "Drug Trafficking":      ["NDPS Act 20 - Production/Sale"],
    "Drug Possession":       ["NDPS Act 27 - Possession"],
    "Weapons Offenses":      ["Arms Act 25 - Unlicensed Arms"],
    "Disorderly Conduct":    ["IPC 268 - Public Nuisance"],
    "Traffic/DUI":           ["MV Act 185 - Drunk Driving", "IPC 279 - Rash Driving"],
    "Hit and Run":           ["MV Act 161 - Hit and Run", "IPC 304A - Death by Negligence"],
    "Stalking":              ["IPC 354D - Stalking"],
    "Harassment":            ["IPC 504 - Intentional Insult", "IPC 506 - Criminal Intimidation"],
    "Robbery":               ["IPC 392 - Robbery", "IPC 394 - Robbery with Hurt"],
}

print("✅ BNS + IPC section mappings loaded")

# ── Department routing ────────────────────────────────────
DEPARTMENT_MAP = {
    "Cybercrime/Hacking":    "Cyber Crime Cell",
    "Identity Theft":        "Cyber Crime Cell",
    "Fraud/Deception":       "Economic Offences Wing",
    "Embezzlement":          "Economic Offences Wing",
    "Drug Trafficking":      "Narcotics Control Bureau",
    "Drug Possession":       "Narcotics Control Bureau",
    "Homicide":              "Criminal Investigation Department",
    "Attempted Murder":      "Criminal Investigation Department",
    "Kidnapping":            "Criminal Investigation Department",
    "Sexual Assault":        "Women Safety Wing",
    "Domestic Violence":     "Women Safety Wing",
    "Stalking":              "Women Safety Wing",
    "Weapons Offenses":      "Special Weapons Task Force",
    "Traffic/DUI":           "Traffic Police",
    "Hit and Run":           "Traffic Police",
}
DEFAULT_DEPARTMENT = "General Duty Officer"

# ── Victim guidance messages ──────────────────────────────
VICTIM_GUIDANCE = {
    "Critical": [
        "Your complaint has been marked CRITICAL — an officer will contact you within 30 minutes.",
        "Please stay in a safe location and do not tamper with any evidence.",
        "If you are in immediate danger, call 112 right now.",
    ],
    "High": [
        "Your complaint has been registered with HIGH priority.",
        "An officer will follow up with you within 2 hours.",
        "Please preserve any evidence — screenshots, call logs, or physical items.",
    ],
    "Medium": [
        "Your complaint has been registered successfully.",
        "An officer will contact you within 24 hours.",
        "Please note your complaint reference number for follow-up.",
    ],
    "Low": [
        "Your complaint has been registered.",
        "An officer will review your complaint within 48 hours.",
        "You can track your complaint status using your reference number.",
    ],
}

def get_ipc_sections(categories: list) -> list:
    # """Returns all relevant IPC sections for the given crime categories."""
    # sections = []
    # for cat in categories:
    #     sections.extend(IPC_SECTIONS.get(cat, []))
    # return list(dict.fromkeys(sections))   # remove duplicates, preserve order

    #---------------------------------------------------------------------------------- UPDATED CODE ---------------------------------------------

 # def get_ipc_sections(categories: list) -> list:
    """
    Returns applicable BNS sections for crimes after July 1 2024.
    IT Act sections are always included for cybercrimes (IT Act was not replaced).
    Also shows old IPC reference for context.
    """
    bns  = []
    ipc  = []

    for cat in categories:
        bns.extend(BNS_SECTIONS.get(cat, []))
        ipc.extend(IPC_SECTIONS.get(cat, []))

    # Remove duplicates
    bns = list(dict.fromkeys(bns))
    ipc = list(dict.fromkeys(ipc))

    # Return BNS as primary, IPC as reference
    result = []
    result.append("── Applicable Law (BNS 2024) ──")
    result.extend(bns)
    result.append("── Old IPC Reference ──")
    result.extend(ipc)

    return result



def get_department(categories: list) -> str:
    """Routes complaint to the most relevant police department."""
    for cat in categories:
        if cat in DEPARTMENT_MAP:
            return DEPARTMENT_MAP[cat]
    return DEFAULT_DEPARTMENT




def get_victim_guidance(severity: str) -> list:
    """Returns guidance messages for the victim based on severity."""
    return VICTIM_GUIDANCE.get(severity, VICTIM_GUIDANCE["Low"])



def generate_fir_draft(original, translated, categories, severity) -> str:
    """Auto-generates a pre-filled FIR draft for the officer."""
    date = datetime.now().strftime("%d-%m-%Y %H:%M")
    return f"""
─────────────────────────────────────────────
        FIRST INFORMATION REPORT (DRAFT)
─────────────────────────────────────────────
Date & Time    : {date}
Reference No   : FIR-{str(uuid.uuid4())[:8].upper()}
Severity       : {severity}

COMPLAINT (Original):
{original}

COMPLAINT (English):
{translated}

CRIME CATEGORIES:
{chr(10).join(f'  • {c}' for c in categories)}
─────────────────────────────────────────────
"""

def generate_html_fir(data: dict) -> str:
    categories   = data.get("categories", [])
    severity     = data.get("severity", "Low")
    confidence   = data.get("confidence", "0%")
    department   = data.get("department", "General Duty Officer")
    bns_sections = data.get("ipc_sections", [])
    guidance     = data.get("victim_guidance", [])
    original     = data.get("original_complaint", "")
    translated   = data.get("translated_complaint", "")
    ref_no       = data.get("reference_no", "FIR-XXXXXXXX")
    timestamp    = datetime.now().strftime("%d %b %Y")
    time_str     = datetime.now().strftime("%H:%M IST")
    conf_val     = float(confidence.replace("%", ""))

    # Severity bar colour
    sev_map = {
        "Critical": ("sev-critical", "Officer contact within 30 minutes"),
        "High":     ("sev-high",     "Officer will contact within 2 hours"),
        "Medium":   ("sev-medium",   "Officer will contact within 24 hours"),
        "Low":      ("sev-low",      "Officer will contact within 48 hours"),
    }
    sev_class, sev_desc = sev_map.get(severity, sev_map["Low"])

    # ETA
    eta_map = {"Critical": "30 minutes", "High": "2 hours", "Medium": "24 hours", "Low": "48 hours"}
    eta = eta_map.get(severity, "48 hours")

    # Category tags
    tags_html = "\n".join([f'<span class="tag">{c}</span>' for c in categories])

    # BNS table rows
    bns_rows = ""
    in_old   = False
    for item in bns_sections:
        if "──" in item or "Applicable Law" in item:
            continue
        if "Old IPC" in item and "──" in item:
            continue
        if item.startswith("── Old IPC"):
            if not in_old:
                bns_rows += '<tr class="divider-row"><td colspan="3">Old IPC Reference — pre-July 2024 cases</td></tr>'
                in_old = True
            continue
        parts    = item.split(" - ", 1)
        code     = parts[0].strip()
        desc     = parts[1].strip() if len(parts) > 1 else ""
        row_cls  = "old-row" if in_old else ""
        bns_rows += f'<tr class="{row_cls}"><td class="bns-code">{code}</td><td>{desc}</td><td class="bns-act">{"IPC 1860" if in_old else "BNS 2024"}</td></tr>'

    # Guidance steps
    steps_html = ""
    for i, g in enumerate(guidance, 1):
        steps_html += f'<div class="step"><span class="step-num">0{i}</span><span class="step-text">{g}</span></div>'

    return f"""<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600&display=swap');
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ background:#efefef; font-family:'Inter',sans-serif; color:#111; padding:40px 20px; font-size:13px; line-height:1.6; -webkit-font-smoothing:antialiased; }}
  .page {{ max-width:700px; margin:0 auto; background:#fff; }}
  .top-strip {{ display:flex; align-items:stretch; border-bottom:1.5px solid #111; }}
  .top-strip-left {{ flex:1; padding:28px 32px; }}
  .doc-label {{ font-size:10px; font-weight:500; letter-spacing:3px; text-transform:uppercase; color:#999; margin-bottom:6px; }}
  .doc-title {{ font-size:24px; font-weight:600; letter-spacing:-0.5px; color:#111; line-height:1.1; }}
  .top-strip-right {{ border-left:1.5px solid #111; padding:28px 32px; min-width:200px; display:flex; flex-direction:column; gap:14px; }}
  .ref-block .tiny {{ font-size:9px; letter-spacing:2px; text-transform:uppercase; color:#999; display:block; margin-bottom:3px; }}
  .ref-block .big {{ font-size:13px; font-weight:600; color:#111; letter-spacing:0.3px; }}
  .severity-bar {{ padding:11px 32px; display:flex; align-items:center; justify-content:space-between; border-bottom:1.5px solid #111; }}
  .sev-critical {{ background:#c0392b; color:#fff; }}
  .sev-high     {{ background:#e67e22; color:#fff; }}
  .sev-medium   {{ background:#2980b9; color:#fff; }}
  .sev-low      {{ background:#27ae60; color:#fff; }}
  .sev-left {{ display:flex; align-items:center; gap:10px; }}
  .sev-dot {{ width:8px; height:8px; border-radius:50%; background:rgba(255,255,255,0.7); flex-shrink:0; }}
  .sev-label {{ font-size:11px; font-weight:600; letter-spacing:2px; text-transform:uppercase; }}
  .sev-right {{ font-size:11px; opacity:0.75; }}
  .two-col {{ display:grid; grid-template-columns:1fr 1fr; border-bottom:1px solid #e0e0e0; }}
  .col-cell {{ padding:22px 32px; }}
  .col-cell+.col-cell {{ border-left:1px solid #e0e0e0; }}
  .section {{ padding:22px 32px; border-bottom:1px solid #e0e0e0; }}
  .sec-label {{ font-size:9px; font-weight:500; letter-spacing:2.5px; text-transform:uppercase; color:#aaa; margin-bottom:12px; }}
  .field {{ margin-bottom:14px; }}
  .field:last-child {{ margin-bottom:0; }}
  .field-label {{ font-size:10px; font-weight:500; letter-spacing:1.5px; text-transform:uppercase; color:#aaa; margin-bottom:4px; }}
  .field-value {{ font-size:13px; color:#111; line-height:1.6; }}
  .field-value.large {{ font-size:15px; font-weight:500; }}
  .complaint-text {{ font-size:13px; line-height:1.75; color:#222; padding:12px 16px; background:#f8f8f8; border-left:2px solid #111; }}
  .tags {{ display:flex; flex-wrap:wrap; gap:6px; }}
  .tag {{ font-size:11px; font-weight:500; padding:4px 11px; border:1px solid #111; color:#111; letter-spacing:0.2px; }}
  .conf-row {{ display:flex; align-items:center; gap:12px; margin-top:6px; }}
  .conf-track {{ flex:1; height:3px; background:#e8e8e8; }}
  .conf-fill {{ height:100%; background:#111; }}
  .conf-pct {{ font-size:14px; font-weight:600; min-width:42px; text-align:right; color:#111; }}
  .bns-table {{ width:100%; border-collapse:collapse; font-size:12px; }}
  .bns-table th {{ font-size:9px; font-weight:500; letter-spacing:2px; text-transform:uppercase; color:#fff; background:#111; padding:7px 12px; text-align:left; }}
  .bns-table td {{ padding:8px 12px; border-bottom:1px solid #ebebeb; vertical-align:top; color:#222; }}
  .bns-table tr:last-child td {{ border-bottom:none; }}
  .bns-table tr:nth-child(even) td {{ background:#fafafa; }}
  .bns-code {{ font-weight:600; font-size:12px; white-space:nowrap; color:#111; }}
  .bns-act {{ font-size:10px; color:#aaa; letter-spacing:0.5px; white-space:nowrap; }}
  .old-row td {{ color:#bbb !important; }}
  .old-row .bns-code {{ color:#bbb !important; }}
  .divider-row td {{ background:#f5f5f5 !important; font-size:9px; letter-spacing:2px; text-transform:uppercase; color:#bbb; padding:5px 12px; }}
  .steps {{ display:grid; gap:8px; }}
  .step {{ display:flex; gap:14px; align-items:flex-start; padding:10px 14px; background:#f8f8f8; }}
  .step-num {{ font-size:12px; font-weight:600; color:#111; min-width:18px; line-height:1.6; flex-shrink:0; }}
  .step-text {{ font-size:12px; color:#333; line-height:1.65; }}
  .helplines {{ display:grid; grid-template-columns:repeat(4,1fr); gap:8px; }}
  .helpline {{ padding:12px; border:1px solid #e0e0e0; text-align:center; }}
  .hl-num {{ font-size:20px; font-weight:600; color:#111; display:block; line-height:1.1; margin-bottom:4px; }}
  .hl-lbl {{ font-size:9px; letter-spacing:1px; text-transform:uppercase; color:#999; }}
  .sig-row {{ display:grid; grid-template-columns:1fr 1fr 1fr; }}
  .sig-cell {{ padding:24px 32px 20px; text-align:center; border-right:1px solid #e0e0e0; border-top:1px solid #e0e0e0; }}
  .sig-cell:last-child {{ border-right:none; }}
  .sig-line {{ height:36px; border-bottom:1px solid #111; margin-bottom:8px; }}
  .sig-lbl {{ font-size:9px; letter-spacing:2px; text-transform:uppercase; color:#aaa; }}
  .footer {{ padding:14px 32px; border-top:1.5px solid #111; display:flex; justify-content:space-between; align-items:center; background:#f8f8f8; }}
  .footer-note {{ font-size:10px; color:#aaa; line-height:1.7; }}
  .footer-ref {{ font-size:10px; color:#aaa; text-align:right; line-height:1.7; }}
  .footer-ref strong {{ display:block; font-size:11px; font-weight:600; color:#555; margin-bottom:2px; }}
</style>
</head>
<body>
<div class="page">

  <div class="top-strip">
    <div class="top-strip-left">
      <div class="doc-label">Cyber Crime Complaint</div>
      <div class="doc-title">First Information<br>Report</div>
    </div>
    <div class="top-strip-right">
      <div class="ref-block"><span class="tiny">Reference No.</span><span class="big">{ref_no}</span></div>
      <div class="ref-block"><span class="tiny">Filed On</span><span class="big">{timestamp}</span></div>
      <div class="ref-block"><span class="tiny">Time</span><span class="big">{time_str}</span></div>
    </div>
  </div>

  <div class="severity-bar {sev_class}">
    <div class="sev-left">
      <div class="sev-dot"></div>
      <span class="sev-label">{severity} Severity</span>
    </div>
    <span class="sev-right">{sev_desc}</span>
  </div>

  <div class="section">
    <div class="sec-label">Complaint</div>
    <div class="field">
      <div class="field-label">Original</div>
      <div class="complaint-text">{original}</div>
    </div>
    <div class="field" style="margin-top:10px">
      <div class="field-label">Translated</div>
      <div class="complaint-text" style="border-left-color:#ccc;color:#444">{translated}</div>
    </div>
  </div>
  <div class="two-col">
    <div class="col-cell">
      <div class="sec-label">Crime Classification</div>
      <div class="tags">{tags_html}</div>
      <div style="margin-top:18px">
        <div class="sec-label">AI Confidence</div>
        <div class="field-label" style="margin-bottom:8px">RAG + Zephyr 7B · {len(knowledge_base)}-case database</div>
        <div class="conf-row">
          <div class="conf-track"><div class="conf-fill" style="width:{conf_val}%"></div></div>
          <div class="conf-pct">{confidence}</div>
        </div>
      </div>
    </div>
    <div class="col-cell">
      <div class="sec-label">Case Details</div>
      <div class="field">
        <div class="field-label">Assigned Department</div>
        <div class="field-value large">{department}</div>
      </div>
      <div class="field">
        <div class="field-label">Response ETA</div>
        <div class="field-value large">{eta}</div>
      </div>
      <div class="field">
        <div class="field-label">Portal</div>
        <div class="field-value">cybercrime.gov.in</div>
      </div>
    </div>
  </div>

  <div class="section">
    <div class="sec-label">Applicable Sections of Law</div>
    <table class="bns-table">
      <thead><tr><th>Section</th><th>Description</th><th>Act</th></tr></thead>
      <tbody>{bns_rows}</tbody>
    </table>
  </div>

  <div class="section">
    <div class="sec-label">Recommended Next Steps</div>
    <div class="steps">{steps_html}</div>
  </div>

  <div class="section">
    <div class="sec-label">Emergency Helplines</div>
    <div class="helplines">
      <div class="helpline"><span class="hl-num">112</span><div class="hl-lbl">Police</div></div>
      <div class="helpline"><span class="hl-num">1930</span><div class="hl-lbl">Cyber Crime</div></div>
      <div class="helpline"><span class="hl-num">181</span><div class="hl-lbl">Women</div></div>
      <div class="helpline"><span class="hl-num">1091</span><div class="hl-lbl">Distress</div></div>
    </div>
  </div>

  <div class="sig-row">
    <div class="sig-cell"><div class="sig-line"></div><div class="sig-lbl">Complainant</div></div>
    <div class="sig-cell"><div class="sig-line"></div><div class="sig-lbl">Investigating Officer</div></div>
    <div class="sig-cell"><div class="sig-line"></div><div class="sig-lbl">Station In-Charge</div></div>
  </div>

  <div class="footer">
    <div class="footer-note">AI-generated draft · Subject to officer verification<br>{timestamp} {time_str} · Confidential</div>
    <div class="footer-ref"><strong>{ref_no}</strong>cybercrime.gov.in</div>
  </div>

</div>
</body>
</html>"""

print("✅ generate_html_fir() loaded with v3 template")
print("✅ Automation layer ready")

✅ BNS + IPC section mappings loaded
✅ generate_html_fir() loaded with v3 template
✅ Automation layer ready


In [28]:
# # ── Full pipeline: translate → classify ──────────────────
# def translate_and_classify(hinglish_text: str) -> dict:
#     english_text = translate_to_english(hinglish_text)
#     result = classify_complaint(english_text)
#     return {
#         "original_hinglish" : hinglish_text,
#         "translated_english": english_text,
#         "classification"    : result
#     }

# # ── Test the full pipeline ────────────────────────────────
# tests = [
#     "Sir mera bank account hack ho gaya aur kisi ne Rs 45,000 nikal liye",
#     "Ek aadmi ne mujhe UPI pe fake collect request bheja aur Rs 22,000 chale gaye",
#     "Mere ghar mein ghuskar laptop aur nakdi chura li gayi",
#     "Neighbour ne mujhpe haath uthaya aur dhamki di",
# ]

# print("=" * 55)
# print("FULL PIPELINE TEST — Translation + Classification")
# print("=" * 55)

# for i, complaint in enumerate(tests, 1):
#     print(f"\n── Test {i} ──────────────────────────────────")
#     out = translate_and_classify(complaint)
#     print(f"  ORIGINAL  : {out['original_hinglish']}")
#     print(f"  TRANSLATED: {out['translated_english']}")
#     print(f"  CATEGORIES: {out['classification'].get('categories', 'N/A')}")
#     print(f"  SEVERITY  : {out['classification'].get('severity', 'N/A')}")

# print("\n" + "=" * 55)
# print("✅ Pipeline test complete")

#--------------------------------------------------------------------------------------------------------------------------- PHASE -2 CODE 


# ── CELL 7 : Full Pipeline ────────────────────────────────
import uuid
import numpy as np
from datetime import datetime

def translate_and_classify(hinglish_text: str, victim_email: str = None) -> dict:
    """
    COMPLETE PIPELINE — from raw Hinglish to full automated response:
    Step 1 → Clean + detect language
    Step 2 → Translate to English
    Step 3 → Classify crime type + severity
    Step 4 → Calculate confidence score
    Step 5 → Route to correct department
    Step 6 → Suggest BNS sections
    Step 7 → Generate FIR draft
    Step 8 → Generate victim guidance
    """

    # Step 1 + 2: Translate
    english_text = translate_to_english(hinglish_text)

    # Step 3: Embed + FAISS search
    new_embed          = embedder.encode([english_text])
    distances, indices = index.search(np.array(new_embed), 2)
    confidence         = round(float(1 / (1 + distances[0][0])) * 100, 1)

    # Step 4: Classify with Zephyr
    classification = classify_complaint(english_text)
    categories     = classification.get("categories", [])
    severity       = classification.get("severity", "Low")

    # Step 5: Generate unique FIR reference number
    ref_no = f"FIR-{str(uuid.uuid4())[:8].upper()}"

    # Step 6–8: Automate department, sections, FIR, guidance
    result = {
        "original_complaint":   hinglish_text,
        "translated_complaint": english_text,
        "categories":           categories,
        "severity":             severity,
        "confidence":           f"{confidence}%",
        "department":           get_department(categories),
        "ipc_sections":         get_ipc_sections(categories),
        "fir_draft":            generate_fir_draft(hinglish_text, english_text, categories, severity),
        "victim_guidance":      get_victim_guidance(severity),
        "reference_no":         ref_no,
    }

    # Step 9: Auto-send email if provided
    # if victim_email:
    #     send_fir_email(victim_email, result)

    if victim_email:
    send_welcome_email(victim_email, result)
    send_fir_email(victim_email, result)
    # Feature 2 — schedule escalation for Critical/High cases
    schedule_escalation(
        ref_no             = ref_no,
        severity           = severity,
        categories         = categories,
        original_complaint = hinglish_text,
        victim_email       = victim_email,
        delay_minutes      = 30
    )

    # return result
    # Feature 3 — register case in status tracker
    register_case(result)
    return result


def print_full_result(result: dict):
    """Pretty prints the complete result for demo purposes."""
    print(f"  ORIGINAL  : {result['original_complaint']}")
    print(f"  TRANSLATED: {result['translated_complaint']}")
    print(f"  CATEGORIES: {result['categories']}")
    print(f"  SEVERITY  : {result['severity']}")
    print(f"  CONFIDENCE: {result['confidence']}")
    print(f"  DEPARTMENT: {result['department']}")
    print(f"  REFERENCE : {result['reference_no']}")
    print(f"  IPC SECTIONS:")
    for s in result["ipc_sections"]:
        print(f"    → {s}")
    print(f"  VICTIM GUIDANCE:")
    for g in result["victim_guidance"]:
        print(f"    • {g}")
    print()
    print(f"{'─'*60}")
    print(f"  FIRST INFORMATION REPORT (DRAFT)")
    print(f"{'─'*60}")
    print(result["fir_draft"])


# ── Test complaints ───────────────────────────────────────
tests = [
    "Sir mera bank account hack ho gaya aur kisi ne Rs 45,000 nikal liye",
    "Ek aadmi ne mujhe UPI pe fake collect request bheja aur Rs 22,000 chale gaye",
    "Mere ghar mein ghuskar laptop aur nakdi chura li gayi",
    "Neighbour ne mujhpe haath uthaya aur dhamki di",
]

for i, complaint in enumerate(tests, 1):
    print(f"\n\n{'='*60}")
    print(f"  TEST {i}")
    print(f"{'='*60}")
    result = translate_and_classify(complaint)
    print_full_result(result)

print("\n✅ Cell 7 pipeline ready")



  TEST 1
  [detected: hinglish]
  ORIGINAL  : Sir mera bank account hack ho gaya aur kisi ne Rs 45,000 nikal liye
  TRANSLATED: Sir my bank account got hacked and someone withdrew Rs 45,000
  CATEGORIES: ['Fraud/Deception']
  SEVERITY  : High
  CONFIDENCE: 53.6%
  DEPARTMENT: Economic Offences Wing
  REFERENCE : FIR-08A81DA0
  IPC SECTIONS:
    → ── Applicable Law (BNS 2024) ──
    → BNS 318 - Cheating
    → BNS 318(4) - Cheating with Delivery
    → ── Old IPC Reference ──
    → IPC 420 - Cheating
    → IPC 415 - Cheating Definition
  VICTIM GUIDANCE:
    • Your complaint has been registered with HIGH priority.
    • An officer will follow up with you within 2 hours.
    • Please preserve any evidence — screenshots, call logs, or physical items.



  TEST 2
  [detected: hinglish]
  ORIGINAL  : Ek aadmi ne mujhe UPI pe fake collect request bheja aur Rs 22,000 chale gaye
  TRANSLATED: A guy sent me a fake collect request on UPI and Rs 22,000 was gone
  CATEGORIES: ['Fraud/Deception']
 

In [29]:
# ── CELL 8 : Email Sender ─────────────────────────────────
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from kaggle_secrets import UserSecretsClient

user_secrets   = UserSecretsClient()
GMAIL_ADDRESS  = user_secrets.get_secret("GMAIL_ADDRESS")
GMAIL_APP_PASS = user_secrets.get_secret("GMAIL_APP_PASS")

def send_fir_email(victim_email: str, data: dict) -> bool:
    try:
        ref_no    = data.get("reference_no", "FIR-XXXXXXXX")
        severity  = data.get("severity", "Low")
        html_body = generate_html_fir(data)

        msg            = MIMEMultipart("alternative")
        msg["Subject"] = f"[{severity} Priority] FIR Registered — Ref: {ref_no}"
        msg["From"]    = f"Cyber Crime FIR Portal <{GMAIL_ADDRESS}>"
        msg["To"]      = victim_email

        plain = f"""
Your complaint has been registered successfully.

Reference No : {ref_no}
Severity     : {severity}
Categories   : {', '.join(data.get('categories', []))}
Department   : {data.get('department', '')}
Translated   : {data.get('translated_complaint', '')}

Track your complaint at: cybercrime.gov.in
Emergency Helplines — Police: 112  |  Cyber Crime: 1930
        """

        msg.attach(MIMEText(plain, "plain"))
        msg.attach(MIMEText(html_body, "html"))

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(GMAIL_ADDRESS, GMAIL_APP_PASS)
            server.sendmail(GMAIL_ADDRESS, victim_email, msg.as_string())

        print(f"✅ FIR email sent successfully to {victim_email}")
        send_welcome_email(victim_email, data)   # ← add this line
        return True

    except Exception as e:
        print(f"❌ Email failed: {e}")
        return False

print("✅ Cell 8 ready — email sender loaded")

def send_welcome_email(victim_email: str, data: dict) -> bool:
    """
    Sends a warm, formal acknowledgement email ALONGSIDE the FIR.
    This arrives as a separate email — friendly tone, not legal.
    """
    try:
        ref_no    = data.get("reference_no", "FIR-XXXXXXXX")
        severity  = data.get("severity", "Low")
        categories = ", ".join(data.get("categories", []))
        timestamp = datetime.now().strftime("%d %B %Y at %H:%M IST")

        msg            = MIMEMultipart("alternative")
        msg["Subject"] = f"We have received your complaint — Ref: {ref_no}"
        msg["From"]    = f"Cyber Crime Portal <{GMAIL_ADDRESS}>"
        msg["To"]      = victim_email

        html_body = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600&display=swap');
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  body {{ background:#f4f4f4; font-family:'Inter',sans-serif; padding:40px 20px; color:#111; }}
  .card {{ max-width:580px; margin:0 auto; background:#fff; border-top:3px solid #111; }}
  .top {{ padding:32px 36px 24px; border-bottom:1px solid #eee; }}
  .tag {{ font-size:9px; font-weight:500; letter-spacing:3px; text-transform:uppercase; color:#999; margin-bottom:10px; }}
  .title {{ font-size:22px; font-weight:600; color:#111; line-height:1.2; }}
  .body {{ padding:28px 36px; }}
  .para {{ font-size:13px; color:#444; line-height:1.8; margin-bottom:18px; }}
  .ref-box {{ background:#f8f8f8; border-left:2px solid #111; padding:14px 18px; margin:20px 0; }}
  .ref-box .lbl {{ font-size:9px; letter-spacing:2px; text-transform:uppercase; color:#999; margin-bottom:4px; }}
  .ref-box .val {{ font-size:16px; font-weight:600; color:#111; }}
  .info-row {{ display:grid; grid-template-columns:1fr 1fr; gap:12px; margin:20px 0; }}
  .info-cell {{ padding:12px 14px; background:#f8f8f8; }}
  .info-cell .lbl {{ font-size:9px; letter-spacing:2px; text-transform:uppercase; color:#999; margin-bottom:4px; }}
  .info-cell .val {{ font-size:13px; font-weight:500; color:#111; }}
  .steps {{ margin:20px 0; }}
  .step {{ display:flex; gap:14px; padding:10px 0; border-bottom:1px solid #f0f0f0; align-items:flex-start; }}
  .step:last-child {{ border-bottom:none; }}
  .step-n {{ font-size:11px; font-weight:600; color:#111; min-width:20px; margin-top:1px; }}
  .step-t {{ font-size:12px; color:#444; line-height:1.6; }}
  .helpline {{ background:#111; color:#fff; padding:16px 18px; margin:20px 0; display:flex; justify-content:space-between; align-items:center; }}
  .hl-left {{ font-size:12px; color:#ccc; }}
  .hl-num {{ font-size:20px; font-weight:600; color:#fff; }}
  .footer {{ padding:18px 36px; border-top:1px solid #eee; background:#f8f8f8; }}
  .footer-text {{ font-size:10px; color:#aaa; line-height:1.8; }}
</style>
</head>
<body>
<div class="card">
  <div class="top">
    <div class="tag">Complaint Acknowledgement</div>
    <div class="title">Your complaint has<br>been received.</div>
  </div>
  <div class="body">
    <p class="para">
      Thank you for reaching out. We have successfully received your cybercrime complaint
      and it has been assigned to our team for review. You will be contacted by an officer
      based on the priority level of your case.
    </p>

    <div class="ref-box">
      <div class="lbl">Your Reference Number</div>
      <div class="val">{ref_no}</div>
    </div>

    <div class="info-row">
      <div class="info-cell">
        <div class="lbl">Filed On</div>
        <div class="val">{timestamp}</div>
      </div>
      <div class="info-cell">
        <div class="lbl">Priority Level</div>
        <div class="val">{severity}</div>
      </div>
      <div class="info-cell">
        <div class="lbl">Crime Category</div>
        <div class="val">{categories}</div>
      </div>
      <div class="info-cell">
        <div class="lbl">FIR Document</div>
        <div class="val">Attached separately</div>
      </div>
    </div>

    <p class="para" style="margin-top:8px">Here is what happens next:</p>
    <div class="steps">
      <div class="step">
        <span class="step-n">01</span>
        <span class="step-t">Your complaint is being reviewed by our AI classification system to determine crime type and severity.</span>
      </div>
      <div class="step">
        <span class="step-n">02</span>
        <span class="step-t">A human officer will be assigned to your case based on the priority level within the response ETA.</span>
      </div>
      <div class="step">
        <span class="step-n">03</span>
        <span class="step-t">You will receive updates on your registered email as the case progresses through each stage.</span>
      </div>
      <div class="step">
        <span class="step-n">04</span>
        <span class="step-t">Keep your reference number <strong>{ref_no}</strong> safe for all future follow-ups and status tracking.</span>
      </div>
    </div>

    <div class="helpline">
      <div class="hl-left">National Cyber Crime Helpline<br><span style="font-size:10px;color:#888">Available 24 × 7</span></div>
      <div class="hl-num">1930</div>
    </div>

    <p class="para" style="color:#888;font-size:12px">
      A separate email with your complete First Information Report (FIR) document
      has also been sent to this address. Please check your inbox.
    </p>
  </div>
  <div class="footer">
    <div class="footer-text">
      This is an automated message from the National Cyber Crime Reporting Portal.<br>
      Please do not reply to this email directly.<br>
      cybercrime.gov.in &nbsp;·&nbsp; {ref_no} &nbsp;·&nbsp; Confidential
    </div>
  </div>
</div>
</body>
</html>"""

        plain = f"""
Your complaint has been received successfully.

Reference No : {ref_no}
Filed On     : {timestamp}
Priority     : {severity}
Category     : {categories}

What happens next:
1. AI classification review
2. Officer assignment based on priority
3. Email updates as case progresses
4. Use {ref_no} for all follow-ups

National Cyber Crime Helpline: 1930 (24x7)
Your FIR document has been sent in a separate email.

cybercrime.gov.in
        """

        msg.attach(MIMEText(plain, "plain"))
        msg.attach(MIMEText(html_body, "html"))

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(GMAIL_ADDRESS, GMAIL_APP_PASS)
            server.sendmail(GMAIL_ADDRESS, victim_email, msg.as_string())

        print(f"✅ Welcome email sent to {victim_email}")
        return True

    except Exception as e:
        print(f"❌ Welcome email failed: {e}")
        return False

print("✅ Welcome email function loaded")


# ── Feature 2 : Severity Escalation Alert ────────────────
import threading
import time

# Set your supervisor email here
SUPERVISOR_EMAIL = "your_supervisor_email@gmail.com"  # ← change this

def send_escalation_email(ref_no: str, severity: str,
                           categories: list, original_complaint: str,
                           victim_email: str):
    """
    Sends escalation alert to supervisor if Critical/High case
    is not acknowledged within the delay window.
    """
    try:
        subject = f"🚨 ESCALATION ALERT — {severity} Case {ref_no}"

        body = f"""
        <html>
        <body style="font-family:Inter,sans-serif;background:#fff;padding:32px;color:#111;">

          <div style="max-width:600px;margin:0 auto;border:2px solid #c0392b;">

            <div style="background:#c0392b;padding:20px 28px;">
              <h1 style="color:#fff;font-size:20px;margin:0;">
                🚨 ESCALATION ALERT — Unacknowledged {severity} Case
              </h1>
            </div>

            <div style="padding:28px;">
              <p style="font-size:14px;color:#555;margin-bottom:20px;">
                This case was filed <b>30 minutes ago</b> and has not been
                acknowledged by any officer. Immediate attention required.
              </p>

              <table style="width:100%;border-collapse:collapse;font-size:13px;">
                <tr style="background:#f8f8f8;">
                  <td style="padding:10px 14px;font-weight:600;width:40%;
                             border:1px solid #e0e0e0;">Reference Number</td>
                  <td style="padding:10px 14px;border:1px solid #e0e0e0;">
                    {ref_no}
                  </td>
                </tr>
                <tr>
                  <td style="padding:10px 14px;font-weight:600;
                             border:1px solid #e0e0e0;">Severity</td>
                  <td style="padding:10px 14px;border:1px solid #e0e0e0;
                             color:#c0392b;font-weight:700;">{severity}</td>
                </tr>
                <tr style="background:#f8f8f8;">
                  <td style="padding:10px 14px;font-weight:600;
                             border:1px solid #e0e0e0;">Categories</td>
                  <td style="padding:10px 14px;border:1px solid #e0e0e0;">
                    {', '.join(categories)}
                  </td>
                </tr>
                <tr>
                  <td style="padding:10px 14px;font-weight:600;
                             border:1px solid #e0e0e0;">Victim Email</td>
                  <td style="padding:10px 14px;border:1px solid #e0e0e0;">
                    {victim_email}
                  </td>
                </tr>
                <tr style="background:#f8f8f8;">
                  <td style="padding:10px 14px;font-weight:600;
                             border:1px solid #e0e0e0;">Original Complaint</td>
                  <td style="padding:10px 14px;border:1px solid #e0e0e0;">
                    {original_complaint}
                  </td>
                </tr>
              </table>

              <div style="margin-top:24px;padding:16px;background:#fff5f5;
                          border-left:4px solid #c0392b;">
                <p style="margin:0;font-size:13px;color:#c0392b;font-weight:600;">
                  ACTION REQUIRED: Please assign this case to an officer
                  immediately and update the case status.
                </p>
              </div>

              <div style="margin-top:20px;font-size:11px;color:#aaa;">
                This is an automated escalation from CyberShield AI System.
                Reference: {ref_no}
              </div>
            </div>

          </div>
        </body>
        </html>
        """

        msg                     = MIMEMultipart("alternative")
        msg["Subject"]          = subject
        msg["From"]             = GMAIL_ADDRESS
        msg["To"]               = SUPERVISOR_EMAIL
        msg.attach(MIMEText(body, "html"))

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(GMAIL_ADDRESS, GMAIL_APP_PASS)
            server.sendmail(GMAIL_ADDRESS, SUPERVISOR_EMAIL, msg.as_string())

        print(f"  🚨 Escalation email sent to supervisor for {ref_no}")

    except Exception as e:
        print(f"  [Escalation email error: {e}]")


def schedule_escalation(ref_no: str, severity: str,
                         categories: list, original_complaint: str,
                         victim_email: str, delay_minutes: int = 30):
    """
    Starts a background timer. If severity is Critical or High,
    sends escalation email to supervisor after delay_minutes.
    Only fires for Critical and High cases.
    """
    if severity not in ["Critical", "High"]:
        print(f"  ℹ️  Severity {severity} — no escalation scheduled")
        return

    delay_seconds = delay_minutes * 60

    def _escalate():
        time.sleep(delay_seconds)
        send_escalation_email(ref_no, severity, categories,
                              original_complaint, victim_email)

    t = threading.Thread(target=_escalate, daemon=True)
    t.start()
    print(f"  ⏱️  Escalation scheduled for {ref_no} in {delay_minutes} mins "
          f"(Severity: {severity})")


# ── Quick test for escalation (5 second delay for testing) ─
def test_escalation():
    print("Testing escalation system with 5 second delay...")
    schedule_escalation(
        ref_no            = "FIR-TEST001",
        severity          = "Critical",
        categories        = ["Cybercrime/Hacking"],
        original_complaint= "Test complaint for escalation",
        victim_email      = "victim@example.com",
        delay_minutes     = 0    # 0 = fires in 5 seconds for testing
    )
    print("Waiting 6 seconds for escalation to fire...")
    time.sleep(6)
    print("✅ Escalation test complete — check supervisor inbox")

print("✅ Feature 2 loaded — Severity Escalation Alert ready")
print(f"   Supervisor email : {SUPERVISOR_EMAIL}")
print(f"   Escalation delay : 30 minutes for Critical/High cases")

✅ Cell 8 ready — email sender loaded
✅ Welcome email function loaded


In [30]:
# Quick test — replace with your own email
test_result = translate_and_classify(
    "Sir mera bank account hack ho gaya aur Rs 45000 nikal gaye",
    victim_email="victim@example.com"   # ← put your real email here
)

print(f"Reference: {test_result['reference_no']}")
print(f"Severity:  {test_result['severity']}")
print(f"Check your inbox now!")

  [detected: hinglish]
✅ FIR email sent successfully to victim@example.com
✅ Welcome email sent to victim@example.com
Reference: FIR-24ADA875
Severity:  Critical
Check your inbox now!


In [ ]:
# import asyncio
# import uvicorn

# app = FastAPI()

# class ComplaintRequest(BaseModel):
#     text: str

# @app.post("/classify")
# async def classify_api(request: ComplaintRequest):
#     result = classify_complaint(request.text)
#     return result

# # 1. Start Tunnel
# public_url = ngrok.connect(8000).public_url
# print(f"\n🚀 API LIVE AT: {public_url}/classify\n")

# # 2. Manual Server Setup (Fixes the RuntimeError)
# config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
# server = uvicorn.Server(config)

# # 3. Apply nest_asyncio and run within the existing loop
# nest_asyncio.apply()
# await server.serve()

# -------------------------------------------------------------------------------------------------- PHASE -2 CODE 
import asyncio
import uvicorn

app = FastAPI()

# class ComplaintRequest(BaseModel):
#     text: str

# @app.post("/classify")
# async def classify_api(request: ComplaintRequest):
#     result = translate_and_classify(request.text)
#     return {
#         "status":               "success",
#         "original_complaint":   result["original_complaint"],
#         "translated_complaint": result["translated_complaint"],
#         "categories":           result["categories"],
#         "severity":             result["severity"],
#         "confidence":           result["confidence"],
#         "department":           result["department"],
#         "ipc_sections":         result["ipc_sections"],
#         "victim_guidance":      result["victim_guidance"],
#         "fir_draft":            result["fir_draft"],
#     }

from pydantic import BaseModel, EmailStr

class ComplaintRequest(BaseModel):
    text:         str
    email:        str = None    # victim email — captured from form
    name:         str = None    # victim name — optional
    phone:        str = None    # victim phone — optional

@app.post("/classify")
async def classify_api(request: ComplaintRequest):
    result = translate_and_classify(
        request.text,
        victim_email=request.email    # ← automatically passed in
    )
    return {
        "status":             "success",
        "reference_no":       result["reference_no"],
        "categories":         result["categories"],
        "severity":           result["severity"],
        "confidence":         result["confidence"],
        "department":         result["department"],
        "ipc_sections":       result["ipc_sections"],
        "victim_guidance":    result["victim_guidance"],
        "email_sent":         True if request.email else False,
    }

# ── Feature 3 : Complaint Status Tracker ─────────────────
from typing import Optional
from datetime import datetime

# In-memory case store — holds all complaints this session
case_store = {}

# Valid status transitions
STATUS_LEVELS = [
    "RECEIVED",
    "ASSIGNED",
    "UNDER_INVESTIGATION",
    "RESOLVED"
]

class StatusUpdate(BaseModel):
    ref_no  : str
    status  : str
    officer : Optional[str] = "System"
    note    : Optional[str] = ""


def register_case(result: dict):
    """
    Called automatically every time a complaint is classified.
    Stores the case in case_store for status tracking.
    """
    ref_no = result.get("reference_no", "UNKNOWN")
    case_store[ref_no] = {
        "ref_no"    : ref_no,
        "categories": result.get("categories", []),
        "severity"  : result.get("severity", "Low"),
        "department": result.get("department", "General Duty"),
        "original"  : result.get("original_complaint", ""),
        "status"    : "RECEIVED",
        "officer"   : "Unassigned",
        "filed_at"  : datetime.now().strftime("%d %b %Y %H:%M IST"),
        "updated_at": datetime.now().strftime("%d %b %Y %H:%M IST"),
        "history"   : [
            {
                "status" : "RECEIVED",
                "by"     : "CyberShield AI",
                "time"   : datetime.now().strftime("%d %b %Y %H:%M IST"),
                "note"   : "Complaint received and FIR generated automatically"
            }
        ]
    }
    print(f"  📋 Case registered in tracker: {ref_no}")


@app.get("/status/{ref_no}")
async def get_status(ref_no: str):
    """
    GET /status/FIR-ABC12345
    Returns full case details and history for a reference number.
    """
    if ref_no not in case_store:
        return {
            "error"  : "Reference number not found",
            "ref_no" : ref_no,
            "message": "Please check your reference number and try again"
        }
    return case_store[ref_no]


@app.post("/update-status")
async def update_status(update: StatusUpdate):
    """
    POST /update-status
    { ref_no, status, officer, note }
    Officer uses this to update case status.
    """
    if update.ref_no not in case_store:
        return {"error": "Reference number not found"}

    if update.status not in STATUS_LEVELS:
        return {
            "error"  : f"Invalid status. Must be one of: {STATUS_LEVELS}"
        }

    case = case_store[update.ref_no]
    case["status"]     = update.status
    case["officer"]    = update.officer
    case["updated_at"] = datetime.now().strftime("%d %b %Y %H:%M IST")
    case["history"].append({
        "status": update.status,
        "by"    : update.officer,
        "time"  : datetime.now().strftime("%d %b %Y %H:%M IST"),
        "note"  : update.note
    })

    return {
        "success": True,
        "ref_no" : update.ref_no,
        "status" : update.status,
        "message": f"Case {update.ref_no} updated to {update.status}"
    }


@app.get("/dashboard")
async def dashboard():
    """
    GET /dashboard
    Returns all cases registered this session.
    Officer-facing view.
    """
    if not case_store:
        return {"message": "No cases registered yet", "total": 0}

    summary = []
    for ref_no, case in case_store.items():
        summary.append({
            "ref_no"    : ref_no,
            "categories": case["categories"],
            "severity"  : case["severity"],
            "department": case["department"],
            "status"    : case["status"],
            "officer"   : case["officer"],
            "filed_at"  : case["filed_at"],
        })

    return {
        "total"     : len(summary),
        "cases"     : summary,
        "breakdown" : {
            "Critical"          : sum(1 for c in summary if c["severity"] == "Critical"),
            "High"              : sum(1 for c in summary if c["severity"] == "High"),
            "Medium"            : sum(1 for c in summary if c["severity"] == "Medium"),
            "Low"               : sum(1 for c in summary if c["severity"] == "Low"),
            "Resolved"          : sum(1 for c in summary if c["status"] == "RESOLVED"),
            "Under_Investigation": sum(1 for c in summary if c["status"] == "UNDER_INVESTIGATION"),
            "Received"          : sum(1 for c in summary if c["status"] == "RECEIVED"),
        }
    }


print("✅ Feature 3 loaded — Status Tracker endpoints ready")
print("   GET  /status/{ref_no}  → check case status")
print("   POST /update-status    → officer updates status")
print("   GET  /dashboard        → all cases this session")